# Bereinigen der Buspassagierdaten

## 1. Bibliotheken importieren

In [1]:
# Standardbibliotheken
from functools import reduce

# PySpark-Bibliotheken
from pyspark.sql import SparkSession, DataFrame
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import IntegerType, FloatType, DateType
from pyspark.sql.functions import (
    col, when, lit, mean, percentile_approx, dayofweek, expr, to_date,
    regexp_extract, concat_ws, lpad, date_add, to_timestamp, date_sub,
    regexp_replace, hour, abs, count, round, sum as spark_sum, last
)

## 2. Spark Session erstellen

In [2]:
# Falls noch keine Session existiert, wird sie erstellt.
spark = SparkSession.builder.appName("Bus Occupancy Cleaning").getOrCreate()

## 3. Einlesen der Parquet-Rohdaten

In [3]:
# Definiere den Pfad zu den verarbeiteten Daten im Azure Data Lake Storage
data_path = 'data/imported_data'

# Lade die Parquet-Datei mit Spark ein
data_raw = spark.read.parquet(data_path)

# Zeige die ersten Zeilen der geladenen Daten an
data_raw.show()

## 4. Umwandlung der Wochentage in Integer-Werte

In [4]:
# Entferne die ursprüngliche Spalte `weekday`, da sie neu berechnet wird
data_weekday_cleaned = data_raw.drop("weekday")

# Konvertiere die `date`-Spalte von einem String in das DateType-Format, falls nötig
data_weekday_cleaned = data_weekday_cleaned.withColumn("date", to_date(col("date"), "dd.MM.yyyy"))

# Korrigiere die Wochentagsnummerierung: Montag = 1, ..., Sonntag = 7
data_weekday_cleaned = data_weekday_cleaned.withColumn("weekday", ((dayofweek(col("date")) + 5) % 7) + 1)

# Füge eine `weekend`-Spalte hinzu, basierend auf `day_type` (Sa = Samstag, So = Sonntag)
data_weekday_cleaned = data_weekday_cleaned.withColumn(
    "weekend",
    when(col("day_type").isin(["Sa", "So"]), 1).otherwise(0)
)

# Entferne die `day_type`-Spalte, da sie nicht mehr benötigt wird
data_weekday_cleaned = data_weekday_cleaned.drop("day_type")

## 5. Anpassung der Zeitformate

In [5]:
# Definiere ein Regex-Muster zur Erkennung eines gültigen Zeitformats "HH:mm:ss"
#    - HH: 00-23 (keine Stunden ≥ 24)
#    - MM: 00-59
#    - SS: 00-59
valid_time_pattern = r"^([01]?[0-9]|2[0-3]):([0-5][0-9]):([0-5][0-9])$"

# Identifiziere Zeilen, die:
#    - Nicht dem gültigen Zeitformat entsprechen ODER
#    - Stundenwerte ≥ 24 enthalten
data_invalid_times = data_weekday_cleaned.filter(
    (col("actual_arrival").isNotNull() & ~col("actual_arrival").rlike(valid_time_pattern)) |
    (col("actual_departure").isNotNull() & ~col("actual_departure").rlike(valid_time_pattern)) |
    (col("planned_arrival_time").isNotNull() & ~col("planned_arrival_time").rlike(valid_time_pattern)) |
    (col("planned_departure_time").isNotNull() & ~col("planned_departure_time").rlike(valid_time_pattern))
)

# Zähle die Anzahl der ungültigen Zeilen
invalid_count = data_invalid_times.count()

print(f"Anzahl der Zeilen mit ungültigem Zeitformat oder Stundenwert >= 24: {invalid_count}")

In [6]:
# Füge eine Spalte `deviation` hinzu:
# - Falls die `actual_arrival`-Spalte ein "*" enthält, wird der Wert auf 1 gesetzt (Abweichung vorhanden)
# - Andernfalls bleibt der Wert 0
data_time_cleaned = data_weekday_cleaned.withColumn(
    "deviation", when(col("actual_arrival").contains("*"), 1).otherwise(0)
)

# Füge eine Spalte `stop_skipped` hinzu:
# - Falls `actual_arrival` in Klammern steht ("(HH:mm:ss)"), wurde die Haltestelle übersprungen → Wert = 1
# - Andernfalls bleibt der Wert 0
data_time_cleaned = data_time_cleaned.withColumn(
    "stop_skipped", when(col("actual_arrival").startswith("(") & col("actual_arrival").endswith(")"), 1).otherwise(0)
)

In [7]:
# Definiere die Spalten, die verarbeitet werden sollen (tatsächliche und geplante Zeiten)
time_columns = ["actual_arrival", "actual_departure", "planned_arrival_time", "planned_departure_time"]

# Extrahiere Stunden, Minuten und Sekunden aus den Zeitstempeln
for time_col in time_columns:
    data_time_cleaned = data_time_cleaned.withColumn(
        f"{time_col}_hour", lpad(regexp_extract(col(time_col), r"(\d{1,2}):(\d{1,2}):(\d{1,2})", 1), 2, "0").cast("int")
    ).withColumn(
        f"{time_col}_minute", lpad(regexp_extract(col(time_col), r"(\d{1,2}):(\d{1,2}):(\d{1,2})", 2), 2, "0")
    ).withColumn(
        f"{time_col}_second", lpad(regexp_extract(col(time_col), r"(\d{1,2}):(\d{1,2}):(\d{1,2})", 3), 2, "0")
    )

# Konvertiere die `date`-Spalte vom Format "dd.MM.yyyy" zu "yyyy-MM-dd"
data_time_cleaned = data_time_cleaned.withColumn("date", to_date(col("date"), "dd.MM.yyyy"))

# Falls die Stunde >= 24 ist, passe das Datum an (verschiebe es um einen Tag nach vorne)
for time_col in time_columns:
    data_time_cleaned = data_time_cleaned.withColumn(
        "date",
        when((col(f"{time_col}_hour") >= 24) & col("date").isNotNull(), date_add(col("date"), 1))
        .otherwise(col("date"))
    )

# Falls die Stunde >= 24 ist, passe den Wochentag entsprechend an (verschiebe ihn um einen Tag nach vorne)
data_time_cleaned = data_time_cleaned.withColumn(
    "weekday",
    when(
        (col("actual_arrival_hour") >= 24) | (col("actual_departure_hour") >= 24) |
        (col("planned_arrival_time_hour") >= 24) | (col("planned_departure_time_hour") >= 24),
        when(col("weekday") == 7, 1).otherwise(col("weekday") + 1)
    ).otherwise(col("weekday"))
)

# Falls die Stunde >= 24 ist, passe die Zeit an, indem du sie auf ein 24-Stunden-Format umrechnest
for time_col in time_columns:
    data_time_cleaned = data_time_cleaned.withColumn(
        time_col,
        when(col(f"{time_col}_hour") >= 24,
             concat_ws(":", lpad((col(f"{time_col}_hour") % 24).cast("string"), 2, "0"),
                       col(f"{time_col}_minute"), col(f"{time_col}_second"))
        ).otherwise(
             concat_ws(":", lpad(col(f"{time_col}_hour").cast("string"), 2, "0"),
                       col(f"{time_col}_minute"), col(f"{time_col}_second"))
        )
    )

# Entferne die Zwischen-Spalten (hour, minute, second), die nur für die Umwandlung benötigt wurden
drop_cols = [f"{col_name}_{suffix}" for col_name in time_columns for suffix in ["hour", "minute", "second"]]
data_time_cleaned = data_time_cleaned.drop(*drop_cols)

# Zeige das bereinigte Ergebnis an
data_time_cleaned.show(truncate=False)

### 6. Anpassen der Datentypen

In [8]:
# Zeigt die Datentypen aller Spalten im DataFrame `data_time_cleaned` an.
# Dies ist nützlich, um sicherzustellen, dass alle Spalten den erwarteten Typ haben,
# insbesondere nach Transformationen wie der Konvertierung von Strings in Datums- oder Integer-Werte.
data_time_cleaned.dtypes

In [9]:
# Konvertiere `planned_arrival_time` und `planned_departure_time` zuerst in TIMESTAMP
data_types_cleaned = data_time_cleaned \
    .withColumn("planned_arrival_time", to_timestamp(concat_ws(" ", col("date"), col("planned_arrival_time")), "yyyy-MM-dd HH:mm:ss")) \
    .withColumn("planned_departure_time", to_timestamp(concat_ws(" ", col("date"), col("planned_departure_time")), "yyyy-MM-dd HH:mm:ss")) \
    .withColumn("date", to_date(col("date"), "yyyy-MM-dd"))  # Stelle sicher, dass `date` im DATE-Format bleibt

# Passe das Datum für `actual_arrival` an:
# Falls `actual_arrival` = 23 Uhr und `planned_arrival_time` = 00 Uhr → Datum um einen Tag zurücksetzen
# Falls `actual_arrival` = 00 Uhr und `planned_arrival_time` = 23 Uhr → Datum um einen Tag nach vorne setzen
data_types_cleaned = data_types_cleaned.withColumn(
    "adjusted_date_arrival",
    when(
        (col("actual_arrival").isNotNull()) & (hour(col("actual_arrival")) == 23) & 
        (col("planned_arrival_time").isNotNull()) & (hour(col("planned_arrival_time")) == 0),  
        date_sub(col("date"), 1)  
    ).when(
        (col("actual_arrival").isNotNull()) & (hour(col("actual_arrival")) == 0) & 
        (col("planned_arrival_time").isNotNull()) & (hour(col("planned_arrival_time")) == 23),  
        date_add(col("date"), 1)  
    ).otherwise(col("date"))  
)

# Passe das Datum für `actual_departure` an:
# Falls `actual_departure` = 23 Uhr und `planned_departure_time` = 00 Uhr → Datum um einen Tag zurücksetzen
# Falls `actual_departure` = 00 Uhr und `planned_departure_time` = 23 Uhr → Datum um einen Tag nach vorne setzen
data_types_cleaned = data_types_cleaned.withColumn(
    "adjusted_date_departure",
    when(
        (col("actual_departure").isNotNull()) & (hour(col("actual_departure")) == 23) & 
        (col("planned_departure_time").isNotNull()) & (hour(col("planned_departure_time")) == 0),  
        date_sub(col("date"), 1)  
    ).when(
        (col("actual_departure").isNotNull()) & (hour(col("actual_departure")) == 0) & 
        (col("planned_departure_time").isNotNull()) & (hour(col("planned_departure_time")) == 23),  
        date_add(col("date"), 1)  
    ).otherwise(col("date"))  
)

# Weise die angepassten Datumswerte `actual_arrival` und `actual_departure` zu
data_types_cleaned = data_types_cleaned \
    .withColumn("actual_arrival", to_timestamp(concat_ws(" ", col("adjusted_date_arrival"), col("actual_arrival")), "yyyy-MM-dd HH:mm:ss")) \
    .withColumn("actual_departure", to_timestamp(concat_ws(" ", col("adjusted_date_departure"), col("actual_departure")), "yyyy-MM-dd HH:mm:ss"))

# Konvertiere numerische Spalten in den richtigen Datentyp (Integer)
numeric_columns = ["boarders", "alighters", "occupancy", "capacity", "stop_time", "delay", "weekday", "weekend"]
for column in numeric_columns:
    data_types_cleaned = data_types_cleaned.withColumn(column, col(column).cast("int"))

# Konvertiere `utilization` von Prozentwerten zu einem Dezimalwert
# Entferne das Prozentzeichen ("%") aus `utilization`
data_types_cleaned = data_types_cleaned.withColumn("utilization", regexp_replace(col("utilization"), "%", ""))  

# Ersetze das Komma (",") durch einen Punkt ("."), um Dezimalwerte korrekt zu formatieren
data_types_cleaned = data_types_cleaned.withColumn("utilization", regexp_replace(col("utilization"), ",", "."))  

# Wandle `utilization` in einen Dezimalwert (zwischen 0 und 1) um
data_types_cleaned = data_types_cleaned.withColumn("utilization", col("utilization").cast("double") / 100)  

# Entferne temporäre Spalten
data_types_cleaned = data_types_cleaned.drop("adjusted_date_arrival", "adjusted_date_departure")

# Zeige das Schema des finalen DataFrames an
data_types_cleaned.printSchema()

## 7. Datenqualität prüfen (Null-/Negativwerte)

In [10]:
def check_missing_values(data_frame):
    total_rows = data_frame.count()  # Bestimme die Gesamtanzahl der Zeilen im DataFrame
    
    # Aggregiere die Anzahl der NULL-Werte für jede Spalte in einem Schritt
    missing_values = data_frame.select([
        F.count(F.when(F.col(c).isNull(), 1)).alias(c) for c in data_frame.columns
    ]).collect()[0].asDict()  # Konvertiere das Ergebnis in ein Dictionary

    # Konvertiere das Dictionary in einen DataFrame
    missing_df = spark.createDataFrame([
        (col_name, missing_count, total_rows) 
        for col_name, missing_count in missing_values.items()
    ], ["Column", "Missing Values", "Total Rows"]).orderBy(F.col("Missing Values").desc())  # Sortiere nach Anzahl fehlender Werte

    return missing_df

# Anwendung der Funktion
missing_info = check_missing_values(data_types_cleaned)
missing_info.show()

In [11]:
def check_negative_values(data_frame):
    # Ermittle alle numerischen Spalten (hier als Beispiel: alle Spalten, deren Typ numerisch ist)
    numeric_columns = [field.name for field in data_frame.schema.fields if field.dataType.simpleString() in ['int', 'double', 'float', 'long']]
    
    # Zähle in jeder numerischen Spalte, wie oft Werte < 0 vorkommen, und speichere die Ergebnisse in einem Dictionary
    negative_counts = data_frame.select([
        F.count(F.when(F.col(c) < 0, 1)).alias(c) for c in numeric_columns
    ]).collect()[0].asDict()
    
    # Erstelle eine Liste mit Tupeln (Spaltenname, negative Wertanzahl) für alle Spalten, bei denen negative Werte vorhanden sind
    negative_values_list = [
        (col_name, neg_count) for col_name, neg_count in negative_counts.items() if neg_count > 0
    ]
    
    # Falls keine negativen Werte gefunden wurden, wird None zurückgegeben
    if not negative_values_list:
        return None
    
    # Ansonsten wird ein DataFrame erstellt, der die Spalten "Column" und "Negative Value Count" enthält
    negative_df = spark.createDataFrame(
        negative_values_list, 
        ["Column", "Negative Value Count"]
    ).orderBy(F.col("Negative Value Count").desc())
    
    return negative_df


# Prüfung auf negative Werte durchführen
negative_info = check_negative_values(data_types_cleaned)

# Ergebnisse ausgeben
if negative_info:
    negative_info.show()
else:
    print("Keine negativen Werte gefunden.")

## 8. Bereinigung fehlender Daten

### a) Spalte(n): planned_arrival_time, planned_departure_time

In [12]:
# Filtere alle Zeilen, bei denen entweder `planned_arrival_time` oder `planned_departure_time` NULL ist,
# und zeige diese an.
data_types_cleaned.filter(col("planned_arrival_time").isNull() | col("planned_departure_time").isNull()).show()

In [13]:
# Zähle die Anzahl der Zeilen, in denen `planned_arrival_time` und `planned_departure_time` identisch sind
identical_count = data_types_cleaned.filter(col("planned_arrival_time") == col("planned_departure_time")).count()

# Bestimme die Gesamtanzahl der Zeilen im DataFrame
total_rows = data_types_cleaned.count()

# Berechne den prozentualen Anteil der Zeilen, bei denen `planned_arrival_time` und `planned_departure_time` gleich sind
percentage_identical = (identical_count / total_rows) * 100

# Ergebnisse ausgeben
print(f"Gesamtanzahl der Zeilen: {total_rows}")
print(f"Anzahl der Zeilen mit identischen geplanten Ankunfts- und Abfahrtszeiten: {identical_count}")
print(f"Prozentualer Anteil: {percentage_identical:.2f}%")

# Filtere alle Zeilen, in denen `planned_arrival_time` und `planned_departure_time` unterschiedlich sind,
# und zeige diese an.
data_types_cleaned.filter(col("planned_arrival_time") != col("planned_departure_time")).show()

In [14]:
# Filtere alle Zeilen, in denen `planned_arrival_time` NULL ist
test = data_types_cleaned.filter(col("planned_arrival_time").isNull())

# Zähle, wie viele dieser Zeilen die Haltestellennamen "Einfahrend" oder "Ausfahrend" enthalten
count = test.filter((col("stop_name") == "Einfahrend") | (col("stop_name") == "Ausfahrend")).count()

# Ausgabe der Anzahl der betroffenen Zeilen
print(f"Anzahl der Zeilen mit NULL-Werten in `planned_arrival_time` und Haltestellennamen 'Einfahrend' oder 'Ausfahrend': {count}")

# Entferne Zeilen, in denen `stop_name` entweder "Einfahrend" oder "Ausfahrend" ist
data_cleaned = data_types_cleaned.filter((col("stop_name") != "Einfahrend") & (col("stop_name") != "Ausfahrend"))

# Zeige die verbleibende Anzahl an NULL-Werten in `planned_arrival_time`
print(f"Verbleibende NULL-Werte in `planned_arrival_time`: {data_cleaned.filter(col('planned_arrival_time').isNull()).count()}")

# Zeige die verbleibende Anzahl an NULL-Werten in `planned_departure_time`
print(f"Verbleibende NULL-Werte in `planned_departure_time`: {data_cleaned.filter(col('planned_departure_time').isNull()).count()}")

In [15]:
# Überprüfe fehlende Werte im bereinigten DataFrame `data_cleaned`
missing_info = check_missing_values(data_cleaned)

# Zeige die Anzahl der fehlenden Werte für jede Spalte an
missing_info.show()

In [16]:
# Zeige alle Zeilen, in denen `boarders` NULL ist
null_row = data_cleaned.filter(col("boarders").isNull()).show()

# Entferne Zeilen, in denen `boarders` NULL ist
data_cleaned = data_cleaned.filter(col("boarders").isNotNull())

# Überprüfe, ob noch NULL-Werte in `boarders` vorhanden sind
print(f"Verbleibende NULL-Werte in `boarders`: {data_cleaned.filter(col('boarders').isNull()).count()}")

### b) Spalte(n): delay

In [17]:
# Zeige alle Zeilen, in denen `delay` NULL ist
null_row = data_cleaned.filter(col("delay").isNull()).show()

# Entferne Zeilen, in denen `delay` NULL ist
data_cleaned = data_cleaned.filter(col("delay").isNotNull())

# Überprüfe, ob noch NULL-Werte in `delay` vorhanden sind
print(f"Verbleibende NULL-Werte in `delay`: {data_cleaned.filter(col('delay').isNull()).count()}")

In [18]:
# Überprüfe erneut die fehlenden Werte im bereinigten DataFrame `data_cleaned`
missing_info = check_missing_values(data_cleaned)

# Zeige die Anzahl der fehlenden Werte für jede Spalte an
missing_info.show()

## 9. Kennzeichnung der einzelnen Fahrten

In [19]:
# 1. Für jede Fahrt (journey) wird die Anzahl der Haltestellen (Zeilen) gezählt
journey_station_counts = data_cleaned.groupBy("line", "date") \
    .agg(F.count("*").alias("station_count"))

# 2. Bestimme Buslinien, die nur eine einzige eindeutige Fahrtrichtung besitzen
lines_one_direction = data_cleaned.groupBy("line") \
    .agg(F.countDistinct("direction").alias("direction_count")) \
    .filter(F.col("direction_count") == 1) \
    .select("line")

# Es werden nur Fahrten berücksichtigt, die zu den oben identifizierten Linien gehören
journey_station_counts = journey_station_counts.join(lines_one_direction, on="line", how="inner")

# 3. Berechne den Median der Haltestellenanzahl für jede Buslinie (mittlere station_count)
line_station_stats = journey_station_counts.groupBy("line").agg(
    F.expr("percentile_approx(station_count, 0.5)").alias("median_station_count")
)

# 4. Füge die berechneten Statistiken zu den Fahrtdaten hinzu
journey_with_stats = journey_station_counts.join(line_station_stats, on="line", how="left")

# Erneut wird der Median der Haltestellenanzahl pro Linie gruppiert berechnet und angezeigt
line_median = journey_with_stats.groupBy("line").agg(
    F.expr("percentile_approx(station_count, 0.5)").alias("median_station_count")
)

line_median.show()

In [20]:
# Erzeuge die neuen Spalten in einem Schritt, um den DataFrame nur einmal zu durchlaufen:
data_cleaned = data_cleaned.select(
    "*",  # Behalte alle bestehenden Spalten bei
    # Wenn die Buslinie "33" oder "34" ist, setze is_circular auf 1 (d.h. Fahrt ist zirkulär), sonst auf 0.
    when((col("line") == "33") | (col("line") == "34"), 1).otherwise(0).alias("is_circular"),
    # Wenn die Buslinie "68" ist, setze shuttle_bus auf 1, sonst auf 0.
    when(col("line") == "68", 1).otherwise(0).alias("shuttle_bus"),
    # Wenn die Buslinie in der angegebenen Liste liegt, setze replacement_trip auf 1, sonst auf 0.
    when(col("line").isin("E80", "PW", "E2", "109", "E14", "E81", "180", "114", "117", "E13", "E82", "104", "185", "E7", "E16", "KT2", "E83", "115", "E1", "E17", "108", "101", "KT1", "E15", "E8",
     "105", "E10", "E5", "E11", "E85", "E4", "110", "311", "E9", "106", "98", "P+R", "E12", "112", "E84", "Hä+G", "69", "99", "885"), 1).otherwise(0).alias("replacement_trip")
)

# Boolean mask: Check if 'line' is one of "80".."85"
data_cleaned = data_cleaned.withColumn(
    "line",
    when(col("line").isin(["80", "81", "82", "83", "84", "85"]), F.concat(F.lit("N"), col("line")))
    .otherwise(col("line"))  # Keep other values unchanged
)

# Show distinct values after modification
data_cleaned.select("line").distinct().show(n=100)


In [21]:
# Parameter: Schwellenwert in Sekunden (z.B. 1800 Sekunden = 30 Minuten)
GAP_THRESHOLD = 1800

# Fensterdefinitionen:
# - Für zirkuläre Fahrten: Gruppierung nach device, date, line und is_circular, sortiert nach geplanter Ankunftszeit aufsteigend.
circular_window = Window.partitionBy("device", "date", "line", "is_circular").orderBy(col("planned_arrival_time").asc())
# - Für nicht-zirkuläre Fahrten und zur Füllung von journey_id: Gruppierung nur nach device, sortiert nach geplanter Ankunftszeit aufsteigend.
device_window = Window.partitionBy("device").orderBy(col("planned_arrival_time").asc())

# 1. --- Zirkulärer Fahrtenmarker ---
# Markiere die erste Zeile in jeder zirkulären Fahrten-Gruppe als Start-Haltestelle.
data_with_journey_id = data_cleaned.withColumn("row_number", F.row_number().over(circular_window))
data_with_journey_id = data_with_journey_id.withColumn(
    "starting_station",
    F.when(F.col("row_number") == 1, F.lit(True)).otherwise(F.lit(False))
)
data_with_journey_id = data_with_journey_id.withColumn(
    "circular_marker",
    F.when((F.col("is_circular") == 1) & (F.col("starting_station") == True), F.lit(1)).otherwise(F.lit(0))
)

# 2. --- Nicht-zirkulärer Fahrtenmarker ---
# Bei nicht-zirkulären Fahrten werden Änderungen basierend auf Linien-/Richtungswechseln oder einem großen Zeitabstand erkannt.
data_with_journey_id = data_with_journey_id.withColumn("prev_line", F.lag("line").over(device_window)) \
    .withColumn("prev_direction", F.lag("direction").over(device_window)) \
    .withColumn("prev_timestamp", F.lag("planned_arrival_time").over(device_window)) \
    .withColumn("prev_stop_name", F.lag("stop_name").over(device_window))

# Erzeuge Flag-Spalten für Veränderungen:
data_with_journey_id = data_with_journey_id.withColumn("line_changed", F.when(col("prev_line") != col("line"), 1).otherwise(0))
data_with_journey_id = data_with_journey_id.withColumn("direction_changed", F.when(col("prev_direction") != col("direction"), 1).otherwise(0))
data_with_journey_id = data_with_journey_id.withColumn("time_diff_sec", (F.col("planned_arrival_time").cast("long") - F.col("prev_timestamp").cast("long")))
data_with_journey_id = data_with_journey_id.withColumn("big_time_gap", F.when(col("time_diff_sec") > GAP_THRESHOLD, 1).otherwise(0))
data_with_journey_id = data_with_journey_id.withColumn("same_station_name", F.when(col("stop_name") == col("prev_stop_name"), 1).otherwise(0))

# Definiere den Marker für nicht-zirkuläre Fahrten:
# - Falls für nicht-zirkuläre Fahrten der vorherige Zeitstempel fehlt, setze den Marker auf 1 (Start der Fahrt).
# - Falls ein Linienwechsel, Richtungswechsel, großer Zeitabstand oder ein Wechsel des Haltestellennamens festgestellt wird, setze ebenfalls den Marker auf 1.
data_with_journey_id = data_with_journey_id.withColumn(
    "non_circular_marker",
    F.when((col("is_circular") == 0) & col("prev_timestamp").isNull(), 1)
     .when((col("is_circular") == 0) & ((col("line_changed") + col("direction_changed") + col("big_time_gap") + col("same_station_name")) >= 1), 1)
     .otherwise(0)
)

# 3. --- Kombinierter Marker ---
# Verwende entweder den zirkulären oder den nicht-zirkulären Marker, um eine neue Fahrt zu kennzeichnen.
data_with_journey_id = data_with_journey_id.withColumn(
    "new_journey_marker",
    F.when((F.col("circular_marker") == 1) | (F.col("non_circular_marker") == 1), F.lit(1)).otherwise(F.lit(0))
)

# 4. --- Erzeugung einer Journey ID ---
# Erstelle Hilfsspalten zur Generierung einer eindeutigen Fahrt-ID.
data_with_journey_id = data_with_journey_id.withColumn("date_str", F.date_format(F.col("date"), "yyyyMMdd"))
data_with_journey_id = data_with_journey_id.withColumn("arrival_str", F.date_format(F.col("actual_arrival"), "HHmmss"))
data_with_journey_id = data_with_journey_id.withColumn(
    "new_journey_id",
    F.when(
        F.col("new_journey_marker") == 1,
        F.concat_ws("_", F.col("device"), F.col("date_str"), F.col("arrival_str"), F.col("line"), F.col("direction"))
    )
)

# Fülle die journey_id bedingt:
# - Bei zirkulären Fahrten wird der letzte (nicht-null) new_journey_id-Wert innerhalb des circular_window verwendet.
# - Andernfalls wird der letzte (nicht-null) new_journey_id-Wert innerhalb des device_window verwendet.
data_with_journey_id = data_with_journey_id.withColumn(
    "journey_id",
    F.when(
        col("is_circular") == 1,
        F.last("new_journey_id", ignorenulls=True).over(circular_window)
    ).otherwise(
        F.last("new_journey_id", ignorenulls=True).over(device_window)
    )
)

# 6. --- Aufräumen ---
# Entferne Zwischenspalten, die nur zur Berechnung der journey_id benötigt wurden.
data_with_journey_id = data_with_journey_id.drop(
    "row_number", "starting_station", "circular_marker", "prev_line", "prev_direction", "prev_timestamp",
    "line_changed", "direction_changed", "time_diff_sec", "big_time_gap", "non_circular_marker",
    "new_journey_marker", "new_journey_id", "date_str", "arrival_str", "prev_stop_name", "same_station_name"
)

# Zähle, wie viele Zeilen keine journey_id haben und gebe die Anzahl aus.
null_count = data_with_journey_id.filter(F.col("journey_id").isNull()).count()
print(f"Number of null journey_id: {null_count}")

# Zeige das resultierende DataFrame an.
data_with_journey_id.show()

## 10. Bereinigung fälchlicherweise negativer Daten

### a) Spalte(n): occupancy

In [22]:
# Zeige alle Zeilen, in denen die `occupancy`-Spalte einen negativen Wert enthält
negative_row = data_with_journey_id.filter(col("occupancy") < 0).show()

In [23]:
# 1. Journeys markieren, die irgendeinen negativen "occupancy"-Wert haben:
# Gruppiere nach "journey_id" und berechne das Maximum eines Wertes, der 1 ist, falls "occupancy" < 0, sonst 0.
journey_flag = data_with_journey_id.groupBy("journey_id").agg(
    F.max(F.when(F.col("occupancy") < 0, F.lit(1)).otherwise(F.lit(0))).alias("has_negative_occupancy")
)

# 2. Den erstellten Flag zurück in die Originaldaten einfügen:
# Hierbei wird per "journey_id" ein Join durchgeführt, sodass jede Zeile den Flag "has_negative_occupancy" erhält.
data_flagged = data_with_journey_id.join(journey_flag, on="journey_id", how="left")

# 3. Fahrten (journeys) mit negativer Belegung entfernen:
# Es werden nur die Zeilen behalten, bei denen "has_negative_occupancy" den Wert 0 hat.
# Anschließend wird die Hilfsspalte "has_negative_occupancy" entfernt.
data_cleaned_negative_occupancy = data_flagged.filter(F.col("has_negative_occupancy") == 0).drop("has_negative_occupancy")

# 4. Überprüfung: Zeige alle Zeilen, bei denen "occupancy" negativ ist (sollte leer sein)
data_cleaned_negative_occupancy.filter(col("occupancy") < 0).show()


### b) Spalte(n): stop_time

In [24]:
# Zeige alle Zeilen an, in denen die `stop_time`-Spalte einen negativen Wert enthält
negative_row = data_cleaned_negative_occupancy.filter(col("stop_time") < 0).show()

In [25]:
# Vorher: Wie viele Journey-IDs gibt es insgesamt?
total_journeys_before = data_cleaned_negative_occupancy.select("journey_id").distinct().count()
print(f"Anzahl Journeys VOR dem Filter: {total_journeys_before}")

# 1) Identifiziere alle Journey-IDs, die gegen "actual_departure >= actual_arrival" verstoßen
#    also wo actual_departure < actual_arrival
invalid_journeys = (
    data_cleaned_negative_occupancy.filter(F.col("actual_departure") < F.col("actual_arrival"))
      .select("journey_id")
      .distinct()
)

# 2) Entferne sämtliche Zeilen mit diesen Journey-IDs aus df
data_cleaned_negative_stop_time = data_cleaned_negative_occupancy.join(invalid_journeys, on="journey_id", how="left_anti")

# Nachher: Wie viele Journey-IDs bleiben übrig?
total_journeys_after = data_cleaned_negative_stop_time.select("journey_id").distinct().count()
print(f"Anzahl Journeys NACH dem Filter: {total_journeys_after}")


In [26]:
# Vorher: Anzahl distinct journey_ids im gesamten DataFrame
total_journeys_before = data_cleaned_negative_stop_time.select("journey_id").distinct().count()
print(f"Journey-IDs VOR dem Filtern: {total_journeys_before}")

# 1) Ermitteln, welche Journey-IDs 'ungültig' sind, 
#    also in mindestens einer Zeile stop_time < 0 haben.
invalid_journeys = (
    data_cleaned_negative_stop_time.filter(F.col("stop_time") < 0)
      .select("journey_id")
      .distinct()
)

# 2) Entfernen Sie ALLE Zeilen dieser invalid_journeys via Anti-Join
data_cleaned_negative_stop_time = data_cleaned_negative_stop_time.join(invalid_journeys, on="journey_id", how="left_anti")

# Nachher: Anzahl distinct journey_ids im gefilterten DataFrame
total_journeys_after = data_cleaned_negative_stop_time.select("journey_id").distinct().count()
print(f"Journey-IDs NACH dem Filtern: {total_journeys_after}")

## 11. Bereinigung von Daten mit übermäßiger Buspassagierbesetzung

In [27]:
tolerance = 1.1

rows_exceeding_before = data_cleaned_negative_stop_time.filter(
    col("occupancy") > col("capacity") * tolerance
).count()

# 1) Identifiziere alle Journey-IDs, bei denen occupancy > capacity
invalid_journeys = (
    data_cleaned_negative_stop_time.filter(F.col("occupancy") > F.col("capacity") * tolerance)
      .select("journey_id")
      .distinct()
)

# 2) Entferne sämtliche Zeilen mit diesen Journey-IDs aus df
data_cleaned_excessive_occupancy = data_cleaned_negative_stop_time.join(invalid_journeys, on="journey_id", how="left_anti")

# Jetzt enthält clean_df nur noch Zeilen ohne 'invalid_journeys'
data_cleaned_excessive_occupancy.show()

# Zähle die Anzahl der Zeilen, die nach der Bereinigung weiterhin die Kapazitätsgrenze überschreiten
rows_exceeding_after = data_cleaned_excessive_occupancy.filter(
    col("occupancy") > col("capacity") * tolerance
).count()

print(f"Anzahl der Zeilen, in denen `occupancy` die `capacity * {tolerance}` überschreitet (nach der Bereinigung): {rows_exceeding_after}")

## 12. Neuberechnung der "utilization"-Spalte

In [28]:
# Entferne die vorhandene `utilization`-Spalte, falls sie existiert
final_data_frame = data_cleaned_excessive_occupancy.drop("utilization")

# Füge eine neue `utilization`-Spalte hinzu:
# - Falls `capacity` > 0, berechne `occupancy / capacity`
# - Falls `capacity` = 0 (um Division durch Null zu vermeiden), setze den Wert auf 0
# - Runde das Ergebnis auf zwei Dezimalstellen
final_data_frame = final_data_frame.withColumn(
    "utilization", 
    round(when(col("capacity") > 0, col("occupancy") / col("capacity")).otherwise(0), 2)
)

## 13. Letzte Überprüfung der Datenqualität

In [29]:
# Überprüfe erneut die fehlenden Werte im bereinigten DataFrame `final_data_frame`
missing_info = check_missing_values(final_data_frame)

# Zeige die Anzahl der fehlenden Werte für jede Spalte an
missing_info.show()

In [30]:
# Prüfung auf negative Werte durchführen
negative_info = check_negative_values(final_data_frame)

# Ergebnisse ausgeben
if negative_info:
    negative_info.show()
else:
    print("Keine negativen Werte gefunden.")

## 14. Export des bereinigten DataFrames als Parquet

In [31]:
# Definiere den Speicherpfad für die bereinigten Daten im Parquet-Format
save_full_path = 'data/clean_data'

# Speichere die bereinigten Daten als Parquet-Datei (Überschreiben des bestehenden Inhalts)
final_data_frame.write.mode("overwrite").parquet(save_full_path)

# Bestätige den erfolgreichen Speichervorgang
print(f"Daten erfolgreich gespeichert unter: {save_full_path}")